# Bag-of-Embeddings Sentiment Walkthrough

This notebook is the interactive route through the embeddings chapter homework. Use it to inspect preprocessing, run one experiment at a time, compare artifacts, and write the final result table.

For repeatable command-line runs, backend checks, sweeps, and smoke tests, use `kaggle_bag_of_embeddings_sentiment.py` in this directory.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Configure Paths and Run Settings

Run this notebook from `chapter_embeddings` or from the repository root. If your shared IMDB files live somewhere else, change `DATA_DIR` before running the training cells.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import platform
import shlex
import subprocess
import sys

try:
    from IPython.display import display
except ImportError:  # pragma: no cover - notebooks normally provide display
    display = print


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


CHAPTER_DIR = find_chapter_dir("kaggle_bag_of_embeddings_sentiment.py", "chapter_embeddings")
SCRIPT = CHAPTER_DIR / "kaggle_bag_of_embeddings_sentiment.py"
DATA_DIR = CHAPTER_DIR / "data"
ARTIFACT_DIR = CHAPTER_DIR / "artifacts" / "notebook_walkthrough"
BACKEND = "tensorflow"
SEED = 1234

print("Python:", platform.python_version())
print("Chapter dir:", CHAPTER_DIR)
print("Data dir:", DATA_DIR)
print("Artifact dir:", ARTIFACT_DIR)
print("Keras backend:", BACKEND)

## 2. Check the Companion Script and Dependencies

The notebook calls the companion script as a subprocess. That keeps command-line and notebook runs on the same implementation.

In [ ]:
def run_companion(*args, check=True):
    env = os.environ.copy()
    env["KERAS_BACKEND"] = BACKEND
    command = [sys.executable, str(SCRIPT), *map(str, args)]
    print("$", shlex.join(command))
    result = subprocess.run(
        command,
        cwd=CHAPTER_DIR,
        env=env,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check:
        result.check_returncode()
    return result


run_companion("--check-deps", "--backend", BACKEND)

## 3. Check the Shared IMDB Files

The preferred layout is `data/aclImdb/` from the public ACL IMDB download. A converted `labeledTrainData.tsv` is also accepted, and Kaggle's compatible `testData.tsv` is needed only for optional submission work.

In [ ]:
import pandas as pd

acl_dir = DATA_DIR / "aclImdb"
train_tsv = DATA_DIR / "labeledTrainData.tsv"
test_tsv = DATA_DIR / "testData.tsv"
real_data_available = acl_dir.exists() or train_tsv.exists()

print("ACL IMDB folder:", acl_dir, "exists=" + str(acl_dir.exists()))
print("Training file:", train_tsv, "exists=" + str(train_tsv.exists()))
print("Test file:", test_tsv, "exists=" + str(test_tsv.exists()))

if train_tsv.exists():
    preview = pd.read_csv(train_tsv, sep="\t", quoting=3, nrows=3)
    display(preview[["id", "sentiment", "review"]])
elif acl_dir.exists():
    print("ACL train positives:", len(list((acl_dir / "train" / "pos").glob("*.txt"))))
    print("ACL train negatives:", len(list((acl_dir / "train" / "neg").glob("*.txt"))))
else:
    print("Shared IMDB data is not present yet. Use the synthetic quick run below until the real files are available.")

## 4. Inspect Tokenization and Vectorization

This cell imports helper functions from the script. Token ID `0` is padding and token ID `1` is out-of-vocabulary.

In [ ]:
import numpy as np

spec = importlib.util.spec_from_file_location("embeddings_companion", SCRIPT)
embeddings_companion = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embeddings_companion)

sample_reviews = [
    "A warm, funny movie with excellent acting.",
    "Not a good movie: dull, dull, dull.",
    "Good, not boring, and surprisingly smart.",
]

for review in sample_reviews:
    print(review)
    print(embeddings_companion.clean_and_tokenize(review))

vocab = embeddings_companion.build_vocabulary(sample_reviews, vocab_size=12)
x = embeddings_companion.vectorize_texts(sample_reviews, vocab, max_length=8, np=np)

print("Vocabulary:", vocab)
print("Vectorized shape:", x.shape)
print(x)

## 5. Run a Synthetic Smoke Experiment

Use this before the real IMDB files are available or after changing environments. It confirms that Keras can train the required architecture.

In [ ]:
quick_dir = ARTIFACT_DIR / "quick_synthetic"

run_companion(
    "--synthetic-data",
    "--quick",
    "--backend",
    BACKEND,
    "--artifact-dir",
    quick_dir,
    "--save-artifacts",
    "--export-validation-mistakes",
    "--nearest-neighbors",
)

quick_summary = json.loads((quick_dir / "run_summary.json").read_text())
quick_summary

## 6. Run the TF-IDF Baseline on Real Data

The homework allows a majority-class or TF-IDF logistic-regression baseline. This cell uses the script's TF-IDF baseline and writes a JSON artifact.

In [ ]:
baseline_dir = ARTIFACT_DIR / "tfidf_baseline"

if real_data_available:
    run_companion(
        "--data-dir",
        DATA_DIR,
        "--baseline-only",
        "--artifact-dir",
        baseline_dir,
        "--save-artifacts",
    )
    tfidf_summary = json.loads((baseline_dir / "tfidf_baseline_summary.json").read_text())
    display(tfidf_summary)
else:
    print("Skipped because shared IMDB data is not present.")

## 7. Train the Bag-of-Embeddings Baseline

This is the required neural model: tokenizer -> Embedding -> GlobalAveragePooling1D -> Dense logit. It records validation metrics, training time, parameter count, validation mistakes, and nearest neighbors.

In [ ]:
neural_dir = ARTIFACT_DIR / "bag_embeddings_baseline"

if real_data_available:
    run_companion(
        "--backend",
        BACKEND,
        "--data-dir",
        DATA_DIR,
        "--epochs",
        "8",
        "--batch-size",
        "128",
        "--vocab-size",
        "20000",
        "--max-length",
        "400",
        "--embedding-dim",
        "64",
        "--run-tfidf-baseline",
        "--export-validation-mistakes",
        "--nearest-neighbors",
        "--artifact-dir",
        neural_dir,
        "--save-artifacts",
    )
    neural_summary = json.loads((neural_dir / "run_summary.json").read_text())
    display(neural_summary)
else:
    print("Skipped because shared IMDB data is not present.")

## 8. Run a Controlled Hyperparameter Sweep

Change one or two settings at a time. The script records sweep summaries as JSON and CSV so the homework table can be filled from measured values.

In [ ]:
sweep_dir = ARTIFACT_DIR / "controlled_sweep"
sweep_json = ARTIFACT_DIR / "sweep_config.json"
sweep_json.parent.mkdir(parents=True, exist_ok=True)

sweep_config = [
    {"run_name": "d64_len400", "embedding_dim": 64, "max_length": 400},
    {"run_name": "d128_len400", "embedding_dim": 128, "max_length": 400},
    {"run_name": "d64_len200", "embedding_dim": 64, "max_length": 200},
]
sweep_json.write_text(json.dumps(sweep_config, indent=2) + "\n", encoding="utf-8")

if real_data_available:
    run_companion(
        "--backend",
        BACKEND,
        "--data-dir",
        DATA_DIR,
        "--sweep-json",
        sweep_json,
        "--run-tfidf-baseline",
        "--artifact-dir",
        sweep_dir,
        "--save-artifacts",
    )
    sweep_results = pd.read_csv(sweep_dir / "sweep_results.csv")
    display(sweep_results[["run_name", "max_length", "embedding_dim", "train_seconds", "trainable_parameters", "validation_accuracy"]])
else:
    print("Skipped because shared IMDB data is not present.")

## 9. Inspect Validation Mistakes and Nearest Neighbors

Use these rows for the required error analysis. Look for negation, sarcasm, mixed evidence, unusual vocabulary, HTML artifacts, very short reviews, and truncation.

In [ ]:
candidate_dirs = [neural_dir, quick_dir]
mistake_dir = next((path for path in candidate_dirs if (path / "validation_mistakes.csv").exists()), None)

if mistake_dir is not None:
    mistakes = pd.read_csv(mistake_dir / "validation_mistakes.csv")
    display(mistakes[["true_label", "predicted_label", "probability_positive", "token_count", "oov_count", "truncated", "review"]].head(5))
else:
    print("No validation_mistakes.csv found yet. Run a neural experiment with --export-validation-mistakes.")

neighbor_dir = next((path for path in candidate_dirs if (path / "nearest_neighbors.csv").exists()), None)
if neighbor_dir is not None:
    neighbors = pd.read_csv(neighbor_dir / "nearest_neighbors.csv")
    display(neighbors[["query_true_label", "query_predicted_label", "neighbor_rank", "neighbor_label", "cosine_similarity", "neighbor_review"]].head(10))

## 10. Build the Homework Result Table

This table scans saved artifacts under `ARTIFACT_DIR`. Use it as the starting point for the table in the written report.

In [ ]:
def result_row(path, summary):
    validation = summary.get("validation", {})
    return {
        "artifact": str(path.relative_to(ARTIFACT_DIR)),
        "model": summary.get("model"),
        "vocab": summary.get("vocab_size_requested"),
        "max_len": summary.get("max_length"),
        "d": summary.get("embedding_dim"),
        "params": summary.get("trainable_parameters"),
        "time": summary.get("train_seconds"),
        "val_acc": validation.get("accuracy"),
    }


rows = []
for summary_path in sorted(ARTIFACT_DIR.glob("**/run_summary.json")):
    rows.append(result_row(summary_path, json.loads(summary_path.read_text())))
for summary_path in sorted(ARTIFACT_DIR.glob("**/tfidf_baseline_summary.json")):
    rows.append(result_row(summary_path, json.loads(summary_path.read_text())))

if rows:
    result_table = pd.DataFrame(rows).sort_values(["model", "val_acc"], ascending=[True, False])
    display(result_table)
else:
    print("No saved summaries found yet.")

## 11. Optional Kaggle Submission

Run this only after selecting one final model from validation evidence. The public leaderboard should not be used as the validation set.

In [ ]:
submission_dir = ARTIFACT_DIR / "selected_submission"

if train_tsv.exists() and test_tsv.exists():
    run_companion(
        "--backend",
        BACKEND,
        "--data-dir",
        DATA_DIR,
        "--make-submission",
        "--artifact-dir",
        submission_dir,
        "--save-artifacts",
    )
    submission_summary = json.loads((submission_dir / "run_summary.json").read_text())
    print("Submission path:", submission_summary.get("submission_path"))
else:
    print("Skipped. This cell requires both labeledTrainData.tsv and testData.tsv.")

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.